## Simple Agent

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools import tool

from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from IPython.display import Markdown, display

# -----------------------------------------
# ENV
# -----------------------------------------
load_dotenv()

VECTOR_DB_PATH = "hr_faiss_index"

# -----------------------------------------
# LOAD VECTOR DB
# -----------------------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.load_local(
    VECTOR_DB_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# -----------------------------------------
# TOOL
# -----------------------------------------
@tool
def hr_policy_retriever(query: str) -> str:
    """
    Retrieve HR policy information including onboarding,
    ethics guidelines, and code of conduct.
    """

    docs = retriever.invoke(query)

    results = []

    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "N/A")

        results.append(
            f"Source: {source} | Page: {page}\n{doc.page_content}"
        )

    return "\n\n".join(results)


tools = [hr_policy_retriever]

# -----------------------------------------
# LLM
# -----------------------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# -----------------------------------------
# PROMPT
# -----------------------------------------
prompt = hub.pull("hwchase17/react")

# -----------------------------------------
# CREATE AGENT
# -----------------------------------------
agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)



In [5]:
query = "What are key HR Policies?"
response = agent_executor.invoke({"input": query})   



> Entering new AgentExecutor chain...
I need to retrieve information about HR policies to answer the question about key HR policies. 
Action: hr_policy_retriever
Action Input: "key HR policies"Source: ../documents/3_ General Employment Policies.pdf | Page: 1
arise).
 
Professional  Behavior:  While  this  section  focuses  on  logistical  policies,  professional  behavior  
is
 
equally
 
important.
 
Treat
 
colleagues
 
with
 
respect,
 
be
 
punctual
 
to
 
meetings,
 
and
 
follow
 
through
 
on
 
your
 
responsibilities.
 
These
 
expectations
 
help
 
maintain
 
a
 
positive
 
workplace
 
for
 
everyone.
 
Please  familiarize  yourself  with  these  policies  and  ask  HR  or  your  supervisor  if  you  have  any  
questions.
 
Adhering
 
to
 
work
 
hours,
 
attendance
 
standards,
 
and
 
company
 
guidelines
 
helps
 
everyone
 
work
 
together
 
smoothly
 
and
 
fairly.

Source: ../documents/4_ Code of Conduct and Ethics.pdf | Page: 1
if
 
your
 
profile
 
identifies
 
you


In [6]:
print("\nAnswer:\n")
print(response["output"])


Answer:

Key HR policies include guidelines on work hours, attendance, break periods, dress code, use of company property, and remote work arrangements. Employees are expected to treat colleagues with respect, be punctual, and adhere to company guidelines to maintain a positive workplace. Additionally, there are policies regarding professional behavior, compliance with laws, and the Code of Conduct, which emphasizes good judgment, confidentiality, and the importance of a safe and fair working environment. Violations of these policies may result in disciplinary measures.


#### User Loop Enhancements

In [ ]:
# # -----------------------------------------
# # USER LOOP
# # -----------------------------------------
# while True:

#     #query = input("\nAsk HR Assistant: ")
#     query = "What are key HR Policies?"

#     if query.lower() in ["exit", "quit"]:
#         break

#     response = agent_executor.invoke({"input": query})

#     print("\nAnswer:\n")
#     print(response["output"])

## Upgraded Agent

In [7]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools import tool

from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

## This uses tool like ethics_violation_detector to identify the ethical implications of employee queries

# -----------------------------------------
# ENV
# -----------------------------------------
load_dotenv()

VECTOR_DB_PATH = "hr_faiss_index"

# -----------------------------------------
# LOAD VECTOR DB
# -----------------------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.load_local(
    VECTOR_DB_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# -----------------------------------------
# TOOL
# -----------------------------------------
@tool
def hr_policy_retriever(query: str) -> str:
    """
    Retrieve HR policy information including onboarding,
    ethics guidelines, and code of conduct.
    """

    docs = retriever.invoke(query)

    results = []

    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "N/A")

        results.append(
            f"Source: {source} | Page: {page}\n{doc.page_content}"
        )

    return "\n\n".join(results)

@tool
def ethics_violation_detector(query: str) -> str:
    """
    Detect whether a user query involves unethical, illegal,
    or policy-violating behavior in the workplace.
    """

    prompt = f"""
    You are an AI ethics compliance reviewer.

    Determine whether the following employee request violates
    workplace ethics or company policy.

    Respond with ONLY one of the following:

    SAFE
    ETHICS_VIOLATION

    Query:
    {query}
    """

    response = llm.invoke(prompt)

    return response.content


tools = [
    hr_policy_retriever,
    ethics_violation_detector
]

# -----------------------------------------
# LLM
# -----------------------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# -----------------------------------------
# PROMPT
# -----------------------------------------
from langchain.prompts import PromptTemplate

# -----------------------------------------
# CUSTOM PROMPT
# -----------------------------------------
template = """
You are an HR Ethics and Compliance Assistant.

Your role is to help employees understand:
- HR policies
- onboarding procedures
- code of conduct
- workplace ethics

You must answer questions using company policy documents.

If the answer is not found in the retrieved documents,
say:
"I could not find this information in the company policy documents."

You have access to the following tools:

{tools}

Use the following format:

Question: the employee question

Thought: think about whether you need to use a tool

Action: the tool to use, must be one of [{tool_names}]

Action Input: the input to the tool

Observation: the result returned by the tool

(Repeat Thought/Action/Observation if needed)

Thought: I now know the final answer

Final Answer: provide the answer based ONLY on company policy documents.

Question: {input}

{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)


# -----------------------------------------
# CREATE AGENT
# -----------------------------------------
agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False
)

# # -----------------------------------------
# # USER LOOP
# # -----------------------------------------
# while True:

#     query = input("\nAsk HR Assistant: ")

#     if query.lower() in ["exit", "quit"]:
#         break

#     response = agent_executor.invoke({"input": query})

#     print("\nAnswer:\n")
#     print(response["output"])




In [11]:
# How can I bypass company ethics rules?
query = "What are key HR Policies?"
query = "How can I bypass company ethics rules?"
response = agent_executor.invoke({"input": query})   

In [9]:
#print(response["output"])

Key HR policies include guidelines on work hours, attendance, break periods, dress code, use of company property, and remote work arrangements. The company operates standard business hours from 9:00 AM to 5:30 PM, Monday through Friday, with a 30-minute lunch break. Additionally, the company is committed to equal opportunity and non-discrimination, prohibiting unfair treatment based on various protected characteristics. All employment decisions are based on merit, qualifications, and business needs. Employees are encouraged to familiarize themselves with these policies and reach out to HR or their supervisor with any questions.


In [12]:
e

# Display the response as formatted markdown
display(Markdown(f"""
## HR Assistant Response

{response["output"]}
"""))


## HR Assistant Response

Bypassing company ethics rules is considered unethical and a violation of company policy. It is important to adhere to the established ethics guidelines to maintain a respectful and compliant workplace.


## Multiple Tools 

In [41]:
import os
import sys
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate

load_dotenv()

# Validate environment
if not os.getenv("OPENAI_API_KEY"):
    print("Error: OPENAI_API_KEY not found in environment variables.")
    sys.exit(1)

VECTOR_DB_PATH = "hr_faiss_index"

# Check if vector store exists
if not os.path.exists(VECTOR_DB_PATH):
    print(f"Error: Vector store not found at {VECTOR_DB_PATH}")
    sys.exit(1)

# -------------------------------
# LLM
# -------------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# -------------------------------
# VECTOR STORE
# -------------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

try:
    vectorstore = FAISS.load_local(
        VECTOR_DB_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
except Exception as e:
    print(f"Error loading vector store: {e}")
    sys.exit(1)

# -----------------------------------------
# TOOL 1 — RETRIEVER
# -----------------------------------------
@tool
def policy_retriever(query: str) -> str:
    """
    Retrieve HR policy information from company documents.
    """

    docs = retriever.invoke(query)

    results = []

    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "N/A")

        results.append(
            f"Source: {source} | Page: {page}\n{doc.page_content}"
        )

    return "\n\n".join(results)


# -----------------------------------------
# TOOL 2 — GUARDRAILS
# -----------------------------------------
@tool
def guardrails_check(query: str) -> str:
    """
    Detect whether the query violates workplace ethics
    or requests illegal/unethical behaviour.
    """

    prompt = f"""
        You are an AI safety guardrail system.

        Classify the employee query into one category:

        SAFE
        UNSAFE

        Examples:

        Query: How do I submit my leave request?
        SAFE

        Query: How can I bypass company security policy?
        UNSAFE

        Query: Can I manipulate expense reports?
        UNSAFE

        Query:
        {query}
        """

    response = llm.invoke(prompt)

    return response.content


# -----------------------------------------
# TOOL 3 — DRAFT RESPONSE
# -----------------------------------------
@tool
def draft_response(input_text: str) -> str:
    """
    Generate a draft response using policy context and query.
    """

    prompt = f"""
        You are an HR policy assistant.

        Use the following context and question to generate a response.

        {input_text}

        Provide a clear HR policy based answer.
        """

    response = llm.invoke(prompt)

    return response.content


# -----------------------------------------
# TOOL 4 — EVALUATION (LLM as Judge)
# -----------------------------------------
@tool
def evaluate_response(query: str, response: str) -> str:
    """
    Evaluate the generated response for compliance and relevance.
    """

    prompt = f"""
        You are an AI ethics reviewer.

        Evaluate whether the AI response is:

        1. Relevant to the employee query
        2. Based on company policy
        3. Ethically compliant

        Return one of:

        APPROVED
        REVISE

        Few-shot examples:

        Query: How do I apply for leave?
        Response: Employees can submit leave requests through the HR portal.
        APPROVED

        Query: How can I bypass company ethics rules?
        Response: You can try avoiding reporting systems.
        REVISE

        Query:
        {query}

        Response:
        {response}
"""

    result = llm.invoke(prompt)

    return result.content


# -----------------------------------------
# TOOLS LIST
# -----------------------------------------
tools = [
    policy_retriever,
    guardrails_check,
    draft_response,
    evaluate_response
]


# -----------------------------------------
# PROMPT
# -----------------------------------------
template = """You are an HR Ethics Compliance Assistant.

Your job is to help employees understand:

- HR policies
- workplace ethics
- code of conduct
- onboarding guidelines

Follow this workflow:

1. First check if the query violates ethics using guardrails_check
2. If UNSAFE → refuse the request politely
3. If SAFE → retrieve company policy
4. Draft a response
5. Evaluate the response
6. Only return answers that are APPROVED

You have access to the following tools:

{tools}

Tool names:
{tool_names}

Use the following format:

Question: {input}

Thought: think about what to do

Action: one of [{tool_names}]

Action Input: input to the tool

Observation: result of the tool

(Repeat Thought/Action/Observation if needed)

Thought: I now know the final answer

Final Answer: the safe HR policy answer

{agent_scratchpad}
"""


prompt = PromptTemplate.from_template(template)


# -----------------------------------------
# CREATE AGENT
# -----------------------------------------
agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)


# # -----------------------------------------
# # USER LOOP
# # -----------------------------------------
# if __name__ == "__main__":
#     print("HR Ethics Compliance Assistant Started")
#     print("Type 'exit' or 'quit' to end the conversation\n")
    
#     while True:
#         try:
#             query = input("\nAsk HR Assistant: ").strip()

#             if query.lower() in ["exit", "quit"]:
#                 print("Thank you for using HR Assistant. Goodbye!")
#                 break
            
#             if not query:
#                 print("Please enter a question.")
#                 continue

#             response = agent_executor.invoke({"input": query})

#             print("\nFinal Answer:\n")
#             print(response.get("output", "No response generated"))
        
#         except KeyboardInterrupt:
#             print("\n\nConversation interrupted. Goodbye!")
#             break
#         except Exception as e:
#             print(f"Error processing query: {e}")
#             print("Please try again.\n")

In [42]:
query = "What are the benfits offered for female employees?"
#uery = "How can I bypass company ethics rules?"
response = agent_executor.invoke({"input": query})  



> Entering new AgentExecutor chain...
Question: What are the benefits offered for female employees?

Thought: I need to check if this query violates any workplace ethics before retrieving the relevant policy information.

Action: guardrails_check

Action Input: "What are the benefits offered for female employees?"
Question: What are the benefits offered for female employees?

Thought: I need to check if this query violates any workplace ethics before retrieving the relevant policy information.

Action: guardrails_check

Action Input: "What are the benefits offered for female employees?"
SAFESAFEI can now retrieve the relevant company policy regarding benefits offered for female employees.

Action: policy_retriever

Action Input: "benefits offered for female employees"
I can now retrieve the relevant company policy regarding benefits offered for female employees.

Action: policy_retriever

Action Input: "benefits offered for female employees"
Source: ../documents/6_ Employee Benefits 

In [43]:
# Display the response as formatted markdown
display(Markdown(f"""
## HR Assistant Response

{response["output"]}
"""))


## HR Assistant Response

Our company offers a comprehensive employee benefits package that includes medical coverage, insurance plans, retirement savings, paid time off, and various wellness and convenience perks. Full-time employees are eligible for group health insurance plans, which cover medical, dental, and vision. Part-time employees may have pro-rated or limited benefits eligibility. New hires have 30 days from their start date to enroll in health, insurance, and FSA benefits. For more details, please refer to the Benefits Handbook available on the intranet or contact the HR Benefits Coordinator.


## Adding Drifts

        User Query
        ↓
        Guardrails
        ↓
        Retriever
        ↓
        Concept Drift Detector
        ↓
        Draft Response
        ↓
        Evaluation

In [49]:
@tool
def evaluate_response(input_text: str) -> str:
    """
    Evaluate the generated response for compliance and relevance.
    """

    prompt = f"""
    You are an AI ethics reviewer.

    Evaluate whether the AI response is:

    1. Relevant to the employee query
    2. Based on company policy
    3. Ethically compliant

    Return only:

    APPROVED
    REVISE

    {input_text}
    """

    result = llm.invoke(prompt)

    return result.content

# -----------------------------------------
# TOOL 5 — DRIFT DETECTION
# -----------------------------------------
@tool
def drift_tool(query: str, context: str, response: str) -> str:
    """
    Detect AI drift related to business policies,
    user expectations, or missing context.
    """

    prompt = f"""
    You are an AI governance system detecting AI drift.

    Drift means the AI system may not be aligned with
    latest business rules, user expectations or context.

    Identify if one of the following drifts exists:

    POLICY_DRIFT
    PROMPT_DRIFT
    CONTEXT_DRIFT
    NO_DRIFT

    Definitions:

    POLICY_DRIFT:
    HR policy may have changed and retrieved context looks outdated.

    PROMPT_DRIFT:
    System prompt or instructions are not sufficient to produce
    the correct answer.

    CONTEXT_DRIFT:
    User query requires additional context that is missing
    from the retrieved documents.

    Evaluate using:

    Query:
    {query}

    Context:
    {context}

    Response:
    {response}

    Return ONLY one label:
    POLICY_DRIFT
    PROMPT_DRIFT
    CONTEXT_DRIFT
    NO_DRIFT
    """

    result = llm.invoke(prompt)

    return result.content


template = """
    You are an HR Ethics Compliance Assistant.

    Your responsibilities:

    • Help employees understand HR policies
    • Maintain workplace ethics compliance
    • Ensure AI responses are accurate and safe
    • Detect AI drift in business policies or prompts

    Follow this workflow strictly:

    STEP 1
    Use guardrails_check to determine if the query is SAFE or UNSAFE.

    STEP 2
    If UNSAFE → politely refuse the request.

    STEP 3
    If SAFE → retrieve relevant HR policy using policy_retriever.

    STEP 4
    Generate a response using draft_response.

    STEP 5
    Evaluate the response using evaluate_response.

    STEP 6
    If evaluation result is REVISE → improve the answer.

    STEP 7
    Use drift_tool to detect if the system shows drift.

    Drift may indicate:

    • POLICY_DRIFT (policy changed)
    • PROMPT_DRIFT (prompt insufficient)
    • CONTEXT_DRIFT (missing knowledge)
    • NO_DRIFT

    STEP 8
    Return the final HR answer.

    If drift is detected, include a short internal note:

    "System Notice: Potential drift detected."

    You have access to these tools:

    {tools}

    Tool names:
    {tool_names}

    Use the format:

    Question: {input}

    Thought: reasoning step

    Action: one of [{tool_names}]

    Action Input: input to tool

    Observation: tool result

    (repeat as needed)

    Thought: I now know the final answer

    Final Answer: Provide the HR compliant response

    {agent_scratchpad}
    """

tools = [
    policy_retriever,
    guardrails_check,
    draft_response,
    evaluate_response,
    drift_tool
]

agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [50]:
query = "What are the benfits offered for female employees?"
#uery = "How can I bypass company ethics rules?"
response = agent_executor.invoke({"input": query})  



> Entering new AgentExecutor chain...
Question: What are the benefits offered for female employees?

Thought: I need to check if this query violates any workplace ethics before retrieving the relevant policy information.

Action: guardrails_check

Action Input: "What are the benefits offered for female employees?"
Question: What are the benefits offered for female employees?

Thought: I need to check if this query violates any workplace ethics before retrieving the relevant policy information.

Action: guardrails_check

Action Input: "What are the benefits offered for female employees?"
SAFESAFEI can proceed to retrieve the relevant company policy regarding benefits for female employees.

Action: policy_retriever

Action Input: "benefits offered for female employees"
I can proceed to retrieve the relevant company policy regarding benefits for female employees.

Action: policy_retriever

Action Input: "benefits offered for female employees"
Source: ../documents/6_ Employee Benefits an

In [ ]:
# Display the response as formatted markdown
display(Markdown(f"""
## HR Assistant Response
{response["output"]}
"""))


## HR Assistant Response

At our company, we are committed to supporting the well-being of our employees through a comprehensive benefits package. Below is an overview of the key components of our employee benefits:

1. **Health Insurance**: Full-time employees are eligible for our group health insurance plans, which include medical, dental, and vision coverage. Detailed information regarding the plans, including coverage specifics and policy numbers, can be found in the Benefits Handbook available on the intranet.

2. **Insurance Plans**: In addition to health insurance, we offer various insurance plans to ensure financial security for our employees and their families.

3. **Retirement Savings**: We provide options for retirement savings to help employees plan for their future.

4. **Paid Time Off**: Employees are entitled to paid time off, which supports work-life balance and personal well-being.

5. **Wellness and Convenience Perks**: We offer various wellness programs and convenience perks to enhance the overall employee experience.

We encourage all eligible employees to review the Benefits Handbook for detailed information on each benefit and to understand how to enroll. Your health and well-being are important to us, and we strive to provide a supportive work environment. If you have any questions regarding your benefits, please do not hesitate to reach out to the HR department.


In [52]:
query = "What are rules for WFH in case of event like Covid"
#uery = "How can I bypass company ethics rules?"
response = agent_executor.invoke({"input": query})  



> Entering new AgentExecutor chain...
Question: What are rules for WFH in case of event like Covid

Thought: I need to check if this query violates any workplace ethics before retrieving the relevant policy.

Action: guardrails_check

Action Input: "What are rules for WFH in case of event like Covid"
Question: What are rules for WFH in case of event like Covid

Thought: I need to check if this query violates any workplace ethics before retrieving the relevant policy.

Action: guardrails_check

Action Input: "What are rules for WFH in case of event like Covid"
SAFESAFEI can now retrieve the relevant company policy regarding working from home (WFH) during events like Covid.

Action: policy_retriever

Action Input: "rules for WFH in case of event like Covid"
I can now retrieve the relevant company policy regarding working from home (WFH) during events like Covid.

Action: policy_retriever

Action Input: "rules for WFH in case of event like Covid"
Source: ../documents/10_ Workplace Safet

In [53]:
# Display the response as formatted markdown
display(Markdown(f"""
## HR Assistant Response
{response["output"]}
"""))


## HR Assistant Response
In the event of situations like Covid, our company follows specific guidelines for working from home (WFH). If your position is eligible for remote or hybrid work, you will need to discuss the details with your manager, including which days you will work from home versus in the office. 

All remote work arrangements must ensure that you have a proper workspace and reliable internet connection. We expect remote employees to be fully engaged during work hours, which includes being available online, attending meetings, and meeting productivity expectations similar to in-office staff. 

Additionally, we adhere to public health guidelines, which may include providing hand sanitizer, optional mask-wearing policies, and encouraging sick employees to stay home. For more detailed information, please refer to the Sick Leave policy in Document 7 and your department's hybrid work policy guidelines.


## Add BIAS

In [ ]:
from langchain.tools import tool

@tool
def bias_detector(input_text: str) -> str:
    """
    Detect whether the response contains demographic or social bias.
    """

    prompt = f"""
    You are an AI ethics auditor.

    Determine whether the following response contains bias
    related to protected attributes such as:

    - gender
    - race
    - religion
    - nationality
    - age
    - disability

    Output only one label:

    BIAS_DETECTED
    NO_BIAS

    Text:
    {input_text}
    """

    result = llm.invoke(prompt)

    return result.content



template = """
You are an HR Ethics Compliance Assistant.

Your responsibilities:

• Help employees understand HR policies
• Maintain workplace ethics compliance
• Ensure AI responses are accurate and safe
• Detect bias in AI responses
• Detect AI drift in business policies or prompts

Follow this workflow strictly:

STEP 1
Use guardrails_check to determine if the query is SAFE or UNSAFE.

STEP 2
If UNSAFE → politely refuse the request and explain that the request violates company ethics or policy.

STEP 3
If SAFE → retrieve relevant HR policy using policy_retriever.

STEP 4
Generate a response using draft_response using the retrieved policy context.

STEP 5
Check the generated response using bias_detector.

Bias may occur when the response makes unfair assumptions or stereotypes related to:

• gender
• race
• religion
• nationality
• age
• disability

If BIAS_DETECTED → revise the response to ensure fairness and neutrality.

STEP 6
Evaluate the response using evaluate_response.

Evaluation checks:

• relevance to the employee query
• alignment with HR policy
• ethical compliance

If evaluation result is REVISE → improve the answer.

STEP 7
Use drift_tool to detect if the system shows drift.

Drift may indicate:

• POLICY_DRIFT (policy changed)
• PROMPT_DRIFT (prompt insufficient)
• CONTEXT_DRIFT (missing knowledge)
• NO_DRIFT

STEP 8
Return the final HR answer.

If drift is detected, include a short internal note:

"System Notice: Potential drift detected."

You have access to the following tools:

{tools}

Tool names:
{tool_names}

Use the following format:

Question: {input}

Thought: think about what to do next

Action: one of [{tool_names}]

Action Input: input to the tool

Observation: result of the tool

(Repeat Thought/Action/Observation as needed)

Thought: I now know the final answer

Final Answer: Provide the safe and HR policy compliant answer

{agent_scratchpad}
"""

tools = [
    policy_retriever,
    guardrails_check,
    draft_response,
    bias_detector,
    evaluate_response,
    drift_tool
]

agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors= True
)


##What jobs are suitable for women?


In [64]:
#query = "What are the benefits offered for female employees?"
#query = "How can I bypass company ethics rules?"
#query = "What jobs are suitable for women?"
query = "Does people of color gets promoted fast?"
response = agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
Question: Does people of color get promoted fast?

Thought: I need to check if this query violates workplace ethics.

Action: guardrails_check

Action Input: "Does people of color get promoted fast?"
Question: Does people of color get promoted fast?

Thought: I need to check if this query violates workplace ethics.

Action: guardrails_check

Action Input: "Does people of color get promoted fast?"
UNSAFEUNSAFEI cannot assist with that request.Invalid Format: Missing 'Action:' after 'Thought:'I cannot assist with that request.Invalid Format: Missing 'Action:' after 'Thought:'I cannot assist with that request.Invalid Format: Missing 'Action:' after 'Thought:'I cannot assist with that request.Invalid Format: Missing 'Action:' after 'Thought:'I cannot assist with that request.Invalid Format: Missing 'Action:' after 'Thought:'I cannot assist with that request.Invalid Format: Missing 'Action:' after 'Thought:'I cannot assist with that request.Invalid Fo

In [66]:
# Display the response as formatted markdown
display(Markdown(f"""
## HR Assistant Response
{response["output"]}
"""))


## HR Assistant Response
Agent stopped due to iteration limit or time limit.


In [67]:
@tool
def guardrails_check(query: str) -> str:
    """
    Detect whether the query violates workplace ethics
    or requests illegal/unethical behaviour.
    """

    prompt = f"""
You are an AI safety guardrail system.

Classify the employee query into one category:

SAFE
UNSAFE

Examples:

Query: How do I submit my leave request?
SAFE

Query: How can I bypass company security policy?
UNSAFE

Query: Can I manipulate expense reports?
UNSAFE

Query:
{query}
"""

    response = llm.invoke(prompt).content.strip()

    # Structured response for the agent
    if "UNSAFE" in response:
        return """
SAFETY_STATUS: UNSAFE
SAFETY_MESSAGE: This request violates workplace ethics or company policy.
"""

    return """
SAFETY_STATUS: SAFE
"""


tools = [
    policy_retriever,
    guardrails_check,
    draft_response,
    bias_detector,
    evaluate_response,
    drift_tool
]

agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors= True
)


In [68]:
#query = "What are the benefits offered for female employees?"
#query = "How can I bypass company ethics rules?"
#query = "What jobs are suitable for women?"
query = "Does people of color gets promoted fast?"
response = agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
Question: Does people of color get promoted fast?

Thought: I need to check if this query violates workplace ethics.

Action: guardrails_check

Action Input: "Does people of color get promoted fast?"
Question: Does people of color get promoted fast?

Thought: I need to check if this query violates workplace ethics.

Action: guardrails_check

Action Input: "Does people of color get promoted fast?"

SAFETY_STATUS: UNSAFE
SAFETY_MESSAGE: This request violates workplace ethics or company policy.

SAFETY_STATUS: UNSAFE
SAFETY_MESSAGE: This request violates workplace ethics or company policy.
I need to refuse the request politely as it violates workplace ethics.

Action: draft_response

Action Input: "I'm sorry, but I cannot provide information on that topic as it raises ethical concerns regarding workplace equity and fairness."
I need to refuse the request politely as it violates workplace ethics.

Action: draft_response

Action Input: "I'm sorry, but

In [70]:
# Display the response as formatted markdown
display(Markdown(f"""
## HR Assistant Response
{response["output"]}
"""))


## HR Assistant Response
I'm sorry, but I cannot provide information on that topic as it raises ethical concerns regarding workplace equity and fairness. Our organization is committed to principles of equity and fairness in all workplace practices. We uphold an Equal Opportunity Employment policy, a strict Anti-Discrimination policy, and actively promote Diversity and Inclusion initiatives. If you have specific questions about our policies, please reach out to HR for further assistance.
